# 02 — Read, join, and report

**Learner scaffold · RetailFlow dataset 1.0.0**

Course lessons: Day 1, Lessons 3, 5, and 6.

Goal: Inspect raw CSV fields, preserve guest sales, and produce a first sales report.

Before running: Run 00 successfully. Teach notebook 03 between the reading and joining sections as Lesson 4.

Writes: No persistent tables; this notebook explores the canonical clean batch.

Follow the playlist module order. Build Buddy will help you create the missing numbered exercise cells. The instructor has a separate completed reference. Step IDs are stable anchors for that authoring work. Saved outputs are blank unless explicitly verified locally; a passed local check does not imply Databricks execution.


## Your notebook grows through Build Buddy

Configuration and instructor utility cells are supplied so you can focus on the learning task. The numbered exercise cells have no solutions. Read the playlist instructions, attempt each cell, run it, explain its output, then ask for a hint if needed. A blank exercise cell is incomplete work. Do not use Run All expecting this scaffold to be a finished pipeline. The course revisits notebooks: Lesson 5 runs 02-02, 02-03 and 02-05; Lesson 6 then runs 02-04.


## Configuration

Use the same assigned names in every notebook. Each notebook defines its own variables and helpers; it does not depend on another notebook’s Python memory. Persistent tables are the handoff between notebooks.


In [0]:
# Edit these four values in EVERY notebook before running it.
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "retailflow_files"
TABLE_PREFIX = "rf_retailflow"

import re
for name in [CATALOG, SCHEMA, VOLUME, TABLE_PREFIX]:
    assert re.fullmatch(r"[a-z][a-z0-9_]*", name), "Use lowercase letters, digits, and underscores; start with a letter."
assert not any(name.startswith("your_") for name in [CATALOG, SCHEMA, VOLUME]), "Replace the example names with your assigned names."
assert TABLE_PREFIX != "rf_yourname", "Choose the table prefix assigned to you."
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/retailflow-v1.0.0"
print("Dataset:", DATASET_ROOT)
print("Table prefix:", f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_")


Dataset: /Volumes/workspace/default/retailflow_files/retailflow-v1.0.0
Table prefix: workspace.default.rf_retailflow_


## Supplied support cells

These utilities keep repeated setup and checks in one place inside this notebook. The Python cleaner deliberately processes this tiny fixture on the driver so beginners can inspect each row. Spark handles DataFrames, joins, summaries, and Delta tables. This is a teaching pattern for 80 sales, not a large-scale ingestion design. The helper definitions below are part of the notebook and can be inspected. Learners will construct selected parts through Build Buddy in the later authoring phase.


In [0]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from datetime import date, datetime
from collections import Counter, defaultdict
import csv, json, re

def read_rows(relative_path):
    with (Path(DATASET_ROOT) / relative_path).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))

def money(value):
    return Decimal(value).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

def business_payload(row):
    return {key: value for key, value in row.items() if key != 'source_row_id'}

def business_records(rows):
    return sorted([business_payload(row) for row in rows], key=lambda row: row['sale_id'])

def dimension_maps():
    return (
        {r['store_id']: r for r in read_rows('data/stores.csv')},
        {r['product_id']: r for r in read_rows('data/products.csv')},
        {r['customer_id']: r for r in read_rows('data/customers.csv')},
    )

def clean_sales(raw_rows):
    """Readable Python cleaning for this tiny fixture; no distributed UDF required."""
    stores, products, customers = dimension_maps()
    candidates, rejected, repaired = [], [], []
    for raw in sorted(raw_rows, key=lambda row: row['source_row_id']):
        row = {key: value.strip() for key, value in raw.items()}
        for key in ['store_id', 'product_id', 'customer_id', 'payment_method']:
            row[key] = row[key].upper()
        if row['customer_id'] == 'UNKNOWN':
            row['customer_id'] = ''
        reason = None
        if any(not row[k] for k in ['sale_id', 'source_row_id', 'sale_date', 'store_id', 'product_id']):
            reason = 'REQUIRED_MISSING'
        if not reason:
            try:
                fmt = '%d/%m/%Y' if '/' in row['sale_date'] else '%Y-%m-%d'
                pattern = r'\d{2}/\d{2}/\d{4}' if '/' in row['sale_date'] else r'\d{4}-\d{2}-\d{2}'
                if not re.fullmatch(pattern, row['sale_date']):
                    raise ValueError('Date format mismatch')
                row['sale_date'] = datetime.strptime(row['sale_date'], fmt).date().isoformat()
            except ValueError:
                reason = 'DATE_INVALID'
        if not reason:
            if row['product_id'] not in products:
                reason = 'PRODUCT_UNKNOWN'
            elif row['store_id'] not in stores:
                reason = 'STORE_UNKNOWN'
            elif row['customer_id'] and row['customer_id'] not in customers:
                reason = 'CUSTOMER_UNKNOWN'
        if not reason:
            try:
                if not re.fullmatch(r'\d+', row['quantity']) or int(row['quantity']) < 1:
                    reason = 'QUANTITY_INVALID'
                elif not re.fullmatch(r'\d+', row['record_version']) or int(row['record_version']) < 1:
                    reason = 'VERSION_INVALID'
                else:
                    row['unit_price'] = format(money(row['unit_price'] or products[row['product_id']]['list_price']), '.2f')
                    row['discount_pct'] = row['discount_pct'] or '0'
                    price, discount = Decimal(row['unit_price']), Decimal(row['discount_pct'])
                    if not price.is_finite() or price <= 0:
                        reason = 'PRICE_INVALID'
                    elif not discount.is_finite() or not 0 <= discount <= 100:
                        reason = 'DISCOUNT_INVALID'
                    elif row['payment_method'] not in ['UPI', 'CARD', 'CASH']:
                        reason = 'PAYMENT_INVALID'
            except (ValueError, InvalidOperation):
                reason = 'NUMBER_INVALID'
        if reason:
            rejected.append(dict(raw, rejection_reason=reason))
        else:
            if row != raw:
                repaired.append(row['source_row_id'])
            candidates.append(row)
    groups = defaultdict(list)
    for row in candidates:
        groups[(row['sale_id'], row['record_version'])].append(row)
    accepted, duplicate_ids = [], []
    raw_by_id = {row['source_row_id']: row for row in raw_rows}
    for group in groups.values():
        if any(business_payload(row) != business_payload(group[0]) for row in group):
            rejected.extend(dict(raw_by_id[row['source_row_id']], rejection_reason='VERSION_CONFLICT') for row in group)
        else:
            accepted.append(group[0])
            duplicate_ids.extend(row['source_row_id'] for row in group[1:])
    return accepted, rejected, duplicate_ids, repaired

def current_versions(rows):
    state = {}
    for row in rows:
        old = state.get(row['sale_id'])
        if old is None or int(row['record_version']) > int(old['record_version']):
            state[row['sale_id']] = row
    return sorted(state.values(), key=lambda row: row['sale_id'])

def visible_returns(stage, sales_rows):
    rows = [r for r in read_rows('data/returns.csv') if stage == 'final' or r['available_batch'] == '01']
    sales = {row['sale_id']: row for row in sales_rows}
    assert len({r['return_id'] for r in rows}) == len(rows)
    returned = Counter()
    for row in rows:
        sale = sales[row['sale_id']]
        assert row['available_batch'] in ['01', '02']
        assert date.fromisoformat(row['return_date']) >= date.fromisoformat(sale['sale_date'])
        quantity = int(row['return_quantity'])
        assert quantity > 0
        returned[row['sale_id']] += quantity
        assert returned[row['sale_id']] <= int(sale['quantity'])
        assert Decimal(row['refund_amount']) == money(quantity * Decimal(sale['unit_price']) * (1 - Decimal(sale['discount_pct']) / 100))
    return rows


In [0]:
from pyspark.sql import functions as F, types as T

SALES_COLUMNS = ['source_row_id', 'sale_id', 'sale_date', 'store_id', 'product_id', 'customer_id', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'record_version']
SALES_SCHEMA = T.StructType([
    T.StructField(name, T.DateType() if name == 'sale_date' else
                  T.IntegerType() if name in ['quantity', 'record_version'] else
                  T.DecimalType(12, 2) if name == 'unit_price' else
                  T.DecimalType(5, 2) if name == 'discount_pct' else T.StringType(), True)
    for name in SALES_COLUMNS
])

def sales_frame(rows):
    typed = []
    for row in rows:
        values = []
        for name in SALES_COLUMNS:
            value = row[name]
            if name == 'sale_date': value = date.fromisoformat(value)
            elif name in ['quantity', 'record_version']: value = int(value)
            elif name in ['unit_price', 'discount_pct']: value = Decimal(value)
            elif name == 'customer_id' and value == '': value = None
            values.append(value)
        typed.append(tuple(values))
    return spark.createDataFrame(typed, SALES_SCHEMA)

def raw_frame(relative_path):
    return spark.read.option('header', True).option('inferSchema', False).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/{relative_path}')

def table_name(suffix):
    assert re.fullmatch(r'[a-z][a-z0-9_]*', suffix)
    return f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{suffix}'

def assert_expected(frame, relative_path, ignore_provenance=False):
    columns = [c for c in frame.columns if not (ignore_provenance and c == 'source_row_id')]
    actual = frame.select(*columns)
    # Read full files before dropping provenance so schema fields match CSV positions.
    expected = spark.read.schema(frame.schema).option('header', True).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/checks/expected/{relative_path}').select(*columns)
    assert actual.exceptAll(expected).limit(1).count() == 0, f'Extra/different rows: {relative_path}'
    assert expected.exceptAll(actual).limit(1).count() == 0, f'Missing rows: {relative_path}'
    print('PASS:', relative_path)

def collected_sales(frame):
    rows = []
    for record in frame.select(*SALES_COLUMNS).collect():
        row = record.asDict()
        row['sale_date'] = row['sale_date'].isoformat()
        row['quantity'] = str(row['quantity'])
        row['record_version'] = str(row['record_version'])
        row['unit_price'] = format(row['unit_price'], '.2f')
        row['discount_pct'] = format(row['discount_pct'].normalize(), 'f')
        row['customer_id'] = row['customer_id'] or ''
        rows.append(row)
    return sorted(rows, key=lambda row: row['sale_id'])

def merge_counts(current, incoming):
    joined = incoming.alias('s').join(current.alias('t'), F.col('s.sale_id') == F.col('t.sale_id'), 'left')
    business_columns = [c for c in SALES_COLUMNS if c != 'source_row_id']
    equal = F.lit(True)
    for name in business_columns:
        equal = equal & F.col(f's.{name}').eqNullSafe(F.col(f't.{name}'))
    conflicts = joined.where((F.col('s.record_version') == F.col('t.record_version')) & ~equal)
    assert conflicts.limit(1).count() == 0, 'Equal-version business conflict: investigate before merge.'
    counts = {'insert': 0, 'update': 0, 'noop': 0}
    for row in joined.select(
        F.when(F.col('t.sale_id').isNull(), 'insert')
         .when(F.col('s.record_version') > F.col('t.record_version'), 'update')
         .otherwise('noop').alias('action')
    ).groupBy('action').count().collect():
        counts[row['action']] = row['count']
    return counts

def merge_into_silver(incoming):
    from delta.tables import DeltaTable
    assert incoming.groupBy('sale_id').count().where('count > 1').limit(1).count() == 0
    target = table_name('silver_sales')
    counts = merge_counts(spark.table(target), incoming)
    (DeltaTable.forName(spark, target).alias('t')
        .merge(incoming.alias('s'), 't.sale_id = s.sale_id')
        .whenMatchedUpdateAll(condition='s.record_version > t.record_version')
        .whenNotMatchedInsertAll().execute())
    return counts


## 02-01: Read rows as strings

**Core concept Vibe:** A schema describes column names and types.

**Build Buddy action:** Read initial sales without automatic type guessing and inspect the first records.

**Expected checkpoint:** 60 rows; all raw fields are strings. Stop here to teach notebook 03 for Lesson 4.


In [0]:
# Build Buddy step 02-01
# Read the initial batch with no automatic type guessing: every field stays text.
initial_raw = raw_frame("data/sales_batch_01.csv")
row_count = initial_raw.count()
print("Rows read:", row_count)
assert row_count == 60, row_count
assert all(dtype == "string" for _, dtype in initial_raw.dtypes), initial_raw.dtypes
print("All raw fields are strings.")
initial_raw.show(3, truncate=False)


Rows read: 60
All raw fields are strings.
+-------------+-------+----------+--------+----------+-----------+--------+----------+------------+--------------+--------------+
|source_row_id|sale_id|sale_date |store_id|product_id|customer_id|quantity|unit_price|discount_pct|payment_method|record_version|
+-------------+-------+----------+--------+----------+-----------+--------+----------+------------+--------------+--------------+
|B01-001      |S0001  |2026-01-01|ST01    |P001      |C001       |1       |80.00     |0           |UPI           |1             |
|B01-002      |S0002  |2026-01-01|ST02    |P002      |C002       |2       |60.00     |5           |CARD          |1             |
|B01-003      |S0003  |2026-01-01|ST03    |P003      |C003       |3       |150.00    |10          |CASH          |1             |
+-------------+-------+----------+--------+----------+-----------+--------+----------+------------+--------------+--------------+
only showing top 3 rows


## 02-02: Cast checked values

**Core concept Vibe:** Validate before turning text into dates or numbers.

**Build Buddy action:** Validate initial sales and create a typed Spark DataFrame.

**Expected checkpoint:** 60 typed sales; 10 guest sales remain valid.


In [0]:
# Build Buddy step 02-02
# Validate every row with the supplied cleaner, then build a typed Spark DataFrame.
accepted, rejected, duplicate_ids, repaired = clean_sales(read_rows("data/sales_batch_01.csv"))
print("Accepted:", len(accepted), "| Rejected:", len(rejected), "| Duplicates:", len(duplicate_ids))
assert len(accepted) == 60 and not rejected and not duplicate_ids

typed_sales = sales_frame(accepted)
assert typed_sales.count() == 60

guest_sales = typed_sales.where(F.col("customer_id").isNull())
print("Guest sales kept (no customer):", guest_sales.count())
assert guest_sales.count() == 10

assert_expected(typed_sales, "initial_sales.csv", ignore_provenance=True)
display(typed_sales.limit(5))


Accepted: 60 | Rejected: 0 | Duplicates: 0
Guest sales kept (no customer): 10
PASS: initial_sales.csv


source_row_id,sale_id,sale_date,store_id,product_id,customer_id,quantity,unit_price,discount_pct,payment_method,record_version
B01-001,S0001,2026-01-01,ST01,P001,C001,1,80.00,0.00,UPI,1
B01-002,S0002,2026-01-01,ST02,P002,C002,2,60.00,5.00,CARD,1
B01-003,S0003,2026-01-01,ST03,P003,C003,3,150.00,10.00,CASH,1
B01-004,S0004,2026-01-01,ST04,P004,C004,1,40.00,0.00,UPI,1
B01-005,S0005,2026-01-01,ST05,P005,C005,2,120.00,0.00,CARD,1


## 02-03: Join by identifiers

**Core concept Vibe:** A left join preserves every sale, including guests.

**Build Buddy action:** Enrich sales with store, product and customer names.

**Expected checkpoint:** 60 enriched sales; S0001 resolves to Asha; S0006 remains present without a customer name.


In [0]:
# Build Buddy step 02-03
# Left joins keep every sale, including guest sales with no customer.
stores_dim = raw_frame("data/stores.csv")
products_dim = raw_frame("data/products.csv")
customers_dim = raw_frame("data/customers.csv")

enriched_sales = (typed_sales
    .join(stores_dim.select("store_id", "store_name"), "store_id", "left")
    .join(products_dim.select("product_id", "product_name"), "product_id", "left")
    .join(customers_dim.select("customer_id", "customer_name"), "customer_id", "left"))
assert enriched_sales.count() == 60, enriched_sales.count()

sample = {row["sale_id"]: row for row in
          enriched_sales.where(F.col("sale_id").isin("S0001", "S0006"))
          .select("sale_id", "store_name", "product_name", "customer_name").collect()}
print("S0001:", sample["S0001"])
print("S0006:", sample["S0006"])
assert sample["S0001"]["customer_name"] == "Asha"
assert sample["S0006"]["customer_name"] is None
print("PASS: 60 enriched sales; S0001 resolves to Asha; S0006 stays a guest sale.")


S0001: Row(sale_id='S0001', store_name='RetailFlow Mumbai', product_name='Rice 1 kg', customer_name='Asha')
S0006: Row(sale_id='S0006', store_name='RetailFlow Chennai', product_name='Notebook A5', customer_name=None)
PASS: 60 enriched sales; S0001 resolves to Asha; S0006 stays a guest sale.


## 02-04: Calculate and group sales

**Core concept Vibe:** Grouping combines records that share a key.

**Build Buddy action:** Calculate discounted sales and summarize them by store.

**Expected checkpoint:** Total net sales before returns = INR 14,456.00; every store agrees with the answer key.


In [0]:
# Build Buddy step 02-04
# Line amounts first: gross, discount, and net per sale.
line_amounts = typed_sales.select(
    "sale_id", "store_id", "quantity",
    (F.col("quantity") * F.col("unit_price")).cast("decimal(12,2)").alias("gross_sales"),
    ((F.col("quantity") * F.col("unit_price")) * (F.col("discount_pct") / 100)).cast("decimal(12,2)").alias("discount_amount"),
).withColumn("net_sales", (F.col("gross_sales") - F.col("discount_amount")).cast("decimal(12,2)"))

# Group the discounted sales by store.
by_store = (line_amounts.groupBy("store_id")
    .agg(F.count(F.lit(1)).alias("sale_count"),
         F.sum("quantity").alias("units_sold"),
         F.sum("gross_sales").alias("gross_sales"),
         F.sum("discount_amount").alias("discount_amount"),
         F.sum("net_sales").alias("net_sales"))
    .join(raw_frame("data/stores.csv").select("store_id", "store_name"), "store_id", "left")
    .orderBy("store_id"))
display(by_store)

# Compare every store against the answer key.
answer_key = {row["store_id"]: row for row in read_rows("checks/expected/initial_by_store.csv")}
assert by_store.count() == len(answer_key) == 10
for row in by_store.collect():
    expected_row = answer_key[row["store_id"]]
    assert row["store_name"] == expected_row["store_name"], row
    assert int(row["sale_count"]) == int(expected_row["sale_count"]), row
    assert int(row["units_sold"]) == int(expected_row["units_sold"]), row
    assert Decimal(row["gross_sales"]) == Decimal(expected_row["gross_sales"]), row
    assert Decimal(row["discount_amount"]) == Decimal(expected_row["discount_amount"]), row
    assert Decimal(row["net_sales"]) == Decimal(expected_row["net_sales"]), row
total_net_sales = by_store.agg(F.sum("net_sales").alias("total")).collect()[0]["total"]
print("Total net sales before returns: INR", total_net_sales)
assert total_net_sales == Decimal("14456.00"), total_net_sales
print("PASS: every store agrees with the answer key.")


store_id,sale_count,units_sold,gross_sales,discount_amount,net_sales,store_name
ST01,6,12,1980.00,122.00,1858.00,RetailFlow Mumbai
ST02,6,12,2140.00,145.00,1995.00,RetailFlow Pune
ST03,6,12,2060.00,137.00,1923.00,RetailFlow Delhi
ST04,6,12,1710.00,126.00,1584.00,RetailFlow Jaipur
ST05,6,12,1050.00,42.00,1008.00,RetailFlow Bengaluru
ST06,6,12,1020.00,72.00,948.00,RetailFlow Chennai
ST07,6,12,1010.00,74.00,936.00,RetailFlow Hyderabad
ST08,6,12,1050.00,53.00,997.00,RetailFlow Kolkata
ST09,6,12,1510.00,112.00,1398.00,RetailFlow Bhubaneswar
ST10,6,12,1940.00,131.00,1809.00,RetailFlow Ahmedabad


Total net sales before returns: INR 14456.00
PASS: every store agrees with the answer key.


## 02-05: Show products with no sales

**Core concept Vibe:** Starting from the product list preserves unsold products.

**Build Buddy action:** Left join product totals onto all 12 products.

**Expected checkpoint:** P012 Desk Lamp appears with zero units sold.


In [0]:
# Build Buddy step 02-05
# Start from the product list so unsold products survive the left join.
product_activity = typed_sales.groupBy("product_id").agg(
    F.count(F.lit(1)).alias("sale_count"),
    F.sum("quantity").alias("units_sold"))

products_with_sales = (raw_frame("data/products.csv").select("product_id", "product_name")
    .join(product_activity, "product_id", "left")
    .withColumn("sale_count", F.coalesce(F.col("sale_count"), F.lit(0)))
    .withColumn("units_sold", F.coalesce(F.col("units_sold"), F.lit(0)))
    .orderBy("product_id"))
display(products_with_sales)

answer_key = {row["product_id"]: row for row in read_rows("checks/expected/initial_by_product.csv")}
assert products_with_sales.count() == len(answer_key) == 12
for row in products_with_sales.collect():
    expected_row = answer_key[row["product_id"]]
    assert row["product_name"] == expected_row["product_name"], row
    assert int(row["sale_count"]) == int(expected_row["sale_count"]), row
    assert int(row["units_sold"]) == int(expected_row["units_sold"]), row
desk_lamp = products_with_sales.where(F.col("product_id") == "P012").collect()[0]
print("P012:", desk_lamp["product_name"], "| units sold:", desk_lamp["units_sold"])
assert desk_lamp["product_name"] == "Desk Lamp" and desk_lamp["units_sold"] == 0
print("PASS: all 12 products listed; P012 Desk Lamp appears with zero units sold.")


product_id,product_name,sale_count,units_sold
P001,Rice 1 kg,6,12
P002,Milk 1 L,6,12
P003,Tea 250 g,6,12
P004,Soap 100 g,6,12
P005,Shampoo 200 mL,6,12
P006,Notebook A5,5,11
P007,Pen Pack,5,10
P008,Water Bottle,5,9
P009,Lunch Box,5,11
P010,Towel,5,10


P012: Desk Lamp | units sold: 0
PASS: all 12 products listed; P012 Desk Lamp appears with zero units sold.
